In [ ]:
# 刘晓波全集 
# https://s3.us-west-1.wasabisys.com/p-library/upload-books/2018-05-22/%E5%88%98%E6%99%93%E6%B3%A2%E6%96%87%E9%9B%86%20-%20%E6%9D%8E%E9%B8%BF%E8%92%99.pdf

In [ ]:
%pip install PyPDF2
%pip install pypdf

In [ ]:
%load_ext autoreload
%autoreload 2

import os
from pathlib import Path
# Change to project root directory (parent of scripts folder)
current_dir = Path.cwd()
if current_dir.name == 'scripts':
    os.chdir(current_dir.parent)


In [ ]:
from lib.data.files import download_pdf_from_url, pdf_to_markdown_by_bookmarks

url = "https://s3.us-west-1.wasabisys.com/p-library/upload-books/2018-05-22/%E5%88%98%E6%99%93%E6%B3%A2%E6%96%87%E9%9B%86%20-%20%E6%9D%8E%E9%B8%BF%E8%92%99.pdf"
pdf_path = "刘晓波文集 - 李洪蒙.pdf"
#download_pdf_from_url(url, pdf_path)
output_dir = "./data_lxb/documents"
pdf_to_markdown_by_bookmarks(pdf_path, output_dir)
#os.remove(pdf_path)


In [ ]:
# Split documents into smaller chunks for processing and indexing
from lib.data.chunker import NaiveChunker

chunker = NaiveChunker(
    input_dir="./data_lxb/documents/",
    output_dir="./data_lxb/chunks/",
    chunk_size=500,
    chunk_overlap=100,
    reload=False
)
chunker.run()

In [ ]:
# Extract titles from documents for title-based search indexing
from lib.data.title_extractor import TitleExtractor

extractor = TitleExtractor(
    input_dir="./data_lxb/documents/",
    output_file="./data_lxb/titles.json"
)
extractor.run()

In [ ]:
# LLM Generate summaries for documents using LLM to create concise representations
# cost $1
from lib.data.summary_extractor import SummaryExtractor
import dotenv
dotenv.load_dotenv()

extractor = SummaryExtractor(
    input_dir="./data_lxb/documents/",
    output_file="./data_lxb/summaries.json",
    max_workers=8,
    limit=None
)
await extractor.run(skip_existing=False)

In [ ]:
# Initialize Elasticsearch client for indexing and searching document chunks
from lib.search.elastic_chunk_index import ElasticWriteClientChunks
elastic_chunk = ElasticWriteClientChunks(
    chunk_index_name="lxb",
    chunks_path="./data_lxb/chunks/",
    contexts_path="./data_lxb/contexts/",
    title_path="./data_lxb/titles.json",
    summaries_path="./data_lxb/summaries.json"
)

In [ ]:
# Clear existing chunk index and rebuild it with all document chunks
elastic_chunk.clear_index()
elastic_chunk.insert_chunks(
    batch_size=1000,
    limit=None,
    skip_existing=True
)

In [ ]:
# Initialize Elasticsearch client for indexing and searching document titles
from lib.search.elastic_title_index import ElasticWriteClientTitles
elastic_title = ElasticWriteClientTitles(
    title_index_name="lxb_titles",
    title_path="./data_lxb/titles.json",
    summaries_path="./data_lxb/summaries.json"
)
elastic_title.clear_index()
elastic_title.insert_titles(
    limit=None,
    skip_existing=True
)